In [1]:
import sys
from pathlib import Path

# Locate project root by walking up until .git is found, then make
# config.py importable regardless of Jupyter's working directory.
_root = Path.cwd().resolve()
while not (_root / ".git").exists():
    if _root == _root.parent:
        raise FileNotFoundError("Could not locate project root (.git not found)")
    _root = _root.parent
sys.path.insert(0, str(_root))

from config import DATA_PROCESSED
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold

df = pd.read_csv(DATA_PROCESSED / "diabetic_cohort_preprocessed.csv")
y = df['readmission_30']
groups = df['patient_id']
X = df.drop(columns=['readmission_30', 'patient_id'])

In [2]:
# Claim 1: Each patient appears exactly once

patient_counts = groups.value_counts()
print(f"Max records per patient: {patient_counts.max()}")
print(f"Patients with >1 record: {(patient_counts > 1).sum()}")
assert patient_counts.max() == 1, "Thesis claim 'each patient appears exactly once' is FALSE for this data"
print("CONFIRMED: each patient appears exactly once.")

Max records per patient: 1
Patients with >1 record: 0
CONFIRMED: each patient appears exactly once.


In [3]:
# Claim 2: zero train/test patient overlap in every group fold

gkf = GroupKFold(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    train_patients = set(groups.iloc[train_idx])
    test_patients = set(groups.iloc[test_idx])
    overlap = train_patients & test_patients
    print(f"Fold {fold}: train={len(train_idx)}, test={len(test_idx)}, patient overlap={len(overlap)}")
    assert len(overlap) == 0, f"LEAKAGE in fold {fold}"
print("\nCONFIRMED: zero patient overlap between train/test in every fold.")

Fold 1: train=49785, test=12447, patient overlap=0
Fold 2: train=49785, test=12447, patient overlap=0
Fold 3: train=49786, test=12446, patient overlap=0
Fold 4: train=49786, test=12446, patient overlap=0
Fold 5: train=49786, test=12446, patient overlap=0

CONFIRMED: zero patient overlap between train/test in every fold.


In [4]:
# Group fold vs StratifiedKFold equivalence

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gkf_sizes = [len(test_idx) for _, test_idx in gkf.split(X, y, groups)]
skf_sizes = [len(test_idx) for _, test_idx in skf.split(X, y)]
print(f"GroupKFold fold sizes: {gkf_sizes}")
print(f"StratifiedKFold fold sizes: {skf_sizes}")

GroupKFold fold sizes: [12447, 12447, 12446, 12446, 12446]
StratifiedKFold fold sizes: [12447, 12447, 12446, 12446, 12446]
